In [0]:
%run ../Utilities/Audit_Utils

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "ent_gf_dev")
dbutils.widgets.text("entity_id", "zipt_markup")

catalog = dbutils.widgets.get("catalog")
entity_id = dbutils.widgets.get("entity_id")
AUDIT_TASK_TABLE = f"{catalog}.audit_ict.audit_task_run"
WATERMARK_TABLE = f"{catalog}.metadata_ict.etl_watermark"

cfg = (
    spark.table(f"{catalog}.metadata_ict.etl_setup_config")
    .filter((F.col("entity_id") == entity_id) & (F.col("target_layer") == "trusted") & F.col("is_active"))
    .first()
)
if cfg is None:
    raise ValueError(f"No active trusted config found for entity_id={entity_id}")
if cfg["load_type"].lower() != "merge":
    raise ValueError(f"Unsupported load_type for trusted layer: {cfg['load_type']}")

source_table = f"{catalog}.{cfg['source_read_from']}"   # ent_gf_dev.ict_foundation.zipt_markup
target_table = f"{catalog}.{cfg['target_table']}"       # ent_gf_dev.ict_trusted.zipt_markup

# Column renames for this entity/layer: {source_column: target_column}
column_map = {
    r["source_column"]: r["target_column"]
    for r in spark.table(f"{catalog}.metadata_ict.column_mapping")
    .filter((F.col("table_name") == cfg["entity_id"]) & (F.col("target_layer") == "trusted") & F.col("is_active"))
    .select("source_column", "target_column").distinct().collect()
}
primary_keys = [column_map.get(c, c) for c in cfg["primary_key_cols"]]   # keys use the renamed columns
partition_column = cfg["partition_column"]
load_type = cfg["load_type"].upper()

In [0]:
import uuid
from datetime import datetime, UTC
from delta.tables import DeltaTable

run_id = str(uuid.uuid4())
source_schema, source_name = source_table.rsplit(".", 1)
target_schema = target_table.rsplit(".", 1)[0]

# Watermark: empty last_load_type = initial load, otherwise incremental on watermark_column (_ingest_ts)
wm = spark.table(WATERMARK_TABLE).filter((F.col("entity_id") == entity_id) & (F.col("layer") == "trusted")).first()
wm_col = wm["watermark_column"]
is_initial = not wm["last_load_type"]
new_load_type = "initial" if is_initial else "incremental"

audit_args = dict(
    table_name=entity_id, run_id=run_id, layer=target_schema.split(".")[-1],
    target_table=target_table, load_type=load_type, source_schema=source_schema, start_ts=datetime.now(UTC),
    watermark_from=wm["last_watermark_value"],
)

try:
    foundation_df = spark.table(source_table)
    max_wm = foundation_df.agg(F.max(wm_col)).first()[0]   # foundation _ingest_ts: one fixed value per ingestion run
    # Initial: all rows up to max(_ingest_ts); incremental: last_watermark_value < _ingest_ts <= max(_ingest_ts)
    window_df = foundation_df.filter(F.col(wm_col) <= F.lit(max_wm))
    if not is_initial:
        window_df = window_df.filter(F.col(wm_col) > F.lit(wm["last_watermark_value"]))

    if not is_initial and window_df.isEmpty():
        # Nothing new since last watermark: skip merge, watermark unchanged
        write_task_audit(spark, build_task_audit_details(
            **audit_args, status="SUCCESS", end_ts=datetime.now(UTC), row_count=0,
            message=f"No new rows in {source_table} after {wm_col} {wm['last_watermark_value']}",
        ), AUDIT_TASK_TABLE)
        print(f"No new rows after {wm_col} {wm['last_watermark_value']}, load skipped")
    else:
        print(f"{new_load_type} load from {source_table}: {wm_col} > {wm['last_watermark_value']} and <= {max_wm}")

        # Rename columns from column_mapping, then add trusted audit columns (_ingest_ts, _ingest_date only)
        source_df = (
            window_df.withColumnsRenamed(column_map)
            .withColumn("_ingest_ts", F.current_timestamp())
            .withColumn("_ingest_date", F.to_date(F.col("_ingest_ts")))
        )

        if not spark.catalog.tableExists(target_table):
            # First run: write directly, no merge needed; CDF on so unified can read changes incrementally
            spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema}")
            writer = source_df.write.format("delta").option("delta.enableChangeDataFeed", "true")
            if partition_column:
                writer = writer.partitionBy(partition_column)
            writer.saveAsTable(target_table)
            rows_written = DeltaTable.forName(spark, target_table).history(1).first()["operationMetrics"]["numOutputRows"]
            rows_read, rows_inserted, rows_updated = rows_written, rows_written, 0
        else:
            # Merge on primary keys
            target = DeltaTable.forName(spark, target_table)
            (
                target.alias("t")
                .merge(source_df.alias("s"), " AND ".join(f"t.`{c}` <=> s.`{c}`" for c in primary_keys))
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
            metrics = target.history(1).first()["operationMetrics"]
            rows_read, rows_inserted, rows_updated = (
                metrics["numSourceRows"], metrics["numTargetRowsInserted"], metrics["numTargetRowsUpdated"]
            )

        # Move watermark: previous = last, last = max(_ingest_ts) from foundation (read before trusted overwrites _ingest_ts)
        spark.sql(
            f"""UPDATE {WATERMARK_TABLE}
                SET previous_watermark_value = last_watermark_value,
                    last_watermark_value = CAST(:max_wm AS TIMESTAMP),
                    last_run_id = :last_run_id,
                    last_load_type = :last_load_type,
                    last_success_ts = current_timestamp(),
                    updated_by = current_user()
                WHERE entity_id = :entity_id AND layer = 'trusted'""",
            args={"max_wm": max_wm, "last_run_id": stable_long_id(run_id),
                  "last_load_type": new_load_type, "entity_id": entity_id},
        )

        write_task_audit(spark, build_task_audit_details(
            **audit_args, status="SUCCESS", end_ts=datetime.now(UTC), watermark_to=max_wm,
            row_count=rows_read, rows_inserted=rows_inserted, rows_updated=rows_updated,
        ), AUDIT_TASK_TABLE)
        print(f"Loaded {source_table} -> {target_table}: {rows_inserted} inserted, {rows_updated} updated, watermark = {max_wm}")

except Exception as e:
    write_task_audit(spark, build_task_audit_details(
        **audit_args, status="FAILED", end_ts=datetime.now(UTC),
        message=str(e)[:2000], rows_updated=None, rows_deleted=None,
    ), AUDIT_TASK_TABLE)
    raise